Downloads and extracts text from the last `DOWNLOAD_LIMIT` presentations about HPAI from the PAFF archive.

In [31]:
# read in the colab secret (add it on the left on the key symbol)
from google.colab import userdata
userdata.get('HF_TOKEN');

In [1]:
%pip install requests beautifulsoup4 docling ollama
#docling-vlm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 13.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 522.6/522.6 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.9/283.9 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.0/94.0 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.8/472.8 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 65.8 MB/s eta 0:00:00
   

In [22]:
%rm ./hpai_pdfs -r

In [3]:
import os
import re
import sys
import time
import random
from urllib.parse import urljoin, urlparse, parse_qs
import requests
from bs4 import BeautifulSoup
import time
import json



from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, PictureDescriptionVlmOptions
from docling.document_converter import DocumentConverter, PdfFormatOption


In [25]:
# Prevent Deep Layout Recursion Crashes
sys.setrecursionlimit(10000)

DOWNLOAD_LIMIT = 30
DOWNLOAD_DIR = "./hpai_pdfs"
OUTPUT_DIR = "./PAFF_extractions"
URLS_JSON = "./url_mapping.json" # Added .json extension for clarity
os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

TARGET_URL = "https://food.ec.europa.eu/horizontal-topics/committees/paff-committees/animal-health-and-welfare/presentations_en"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
}

def generate_clean_filename(href, link_text):
    parsed_url = urlparse(href)
    queries = parse_qs(parsed_url.query)
    if 'filename' in queries and queries['filename']:
        return queries['filename'][0]
    clean_text = re.sub(r'[^a-zA-Z0-9_\-]', '_', link_text.strip())
    clean_text = re.sub(r'_+', '_', clean_text).strip('_')
    return f"{clean_text}.pdf" if clean_text else os.path.basename(parsed_url.path)

# Initialize URL mapping if it doesn't exist or is invalid
if not os.path.exists(URLS_JSON):
    with open(URLS_JSON, "w") as f:
        json.dump({}, f)

print(f"Fetching updates from: {TARGET_URL}")
session = requests.Session()
response = session.get(TARGET_URL, headers=HEADERS)

if response.status_code == 200:
    soup = BeautifulSoup(response.text, "html.parser")
    active_pdf_queue = []

    for anchor in soup.find_all("a", href=True):
        href = anchor["href"]
        link_text = anchor.get_text(strip=True)

        if len(active_pdf_queue) >= DOWNLOAD_LIMIT: break

        if "document/download" in href.lower() or href.lower().endswith(".pdf"):
            if re.search(r"hpai", href, re.IGNORECASE) or re.search(r"hpai", link_text, re.IGNORECASE):
                pdf_url = urljoin(TARGET_URL, href)
                filename = generate_clean_filename(href, link_text)
                local_pdf_path = os.path.join(DOWNLOAD_DIR, filename)

                if os.path.exists(local_pdf_path) and os.path.getsize(local_pdf_path) > 5000:
                    active_pdf_queue.append(local_pdf_path)
                    continue

                time.sleep(1.0 + random.random())
                try:
                    file_response = session.get(pdf_url, headers=HEADERS, allow_redirects=True, timeout=15)
                    if file_response.status_code == 200 and file_response.content.startswith(b"%PDF-"):
                        with open(local_pdf_path, "wb") as f:
                            f.write(file_response.content)

                        # Atomic Update of JSON mapping
                        with open(URLS_JSON, "r+") as f:
                            data = json.load(f)
                            data[filename] = pdf_url
                            f.seek(0)
                            json.dump(data, f, indent=4)
                            f.truncate()

                        active_pdf_queue.append(local_pdf_path)
                        print(f"  [Saved] {filename}")
                except Exception as e:
                    print(f"  [Error] {filename}: {e}")

print(f"\n--- Queue Ready: {len(active_pdf_queue)} files ---")

Fetching updates from: https://food.ec.europa.eu/horizontal-topics/committees/paff-committees/animal-health-and-welfare/presentations_en
  [Saved] reg-com_ahw_20260324_pres-13.pdf
  [Saved] reg-com_ahw_20260324_pres-14.pdf
  [Saved] reg-com_ahw_20260324_pres-15.pdf
  [Saved] reg-com_ahw_20260324_pres-16.pdf
  [Saved] reg-com_ahw_20260324_pres-17.pdf
  [Saved] reg-com_ahw_20260219_pres-09.pdf
  [Saved] reg-com_ahw_20260219_pres-10.pdf
  [Saved] reg-com_ahw_20260219_pres-11.pdf
  [Saved] reg-com_ahw_20260219_pres-12.pdf
  [Saved] reg-com_ahw_20260219_pres-13.pdf
  [Saved] reg-com_ahw_20260219_pres-14.pdf
  [Saved] reg-com_ahw_20260219_pres-15.pdf
  [Saved] reg-com_ahw_20260219_pres-16.pdf
  [Saved] reg-com_ahw_20260219_pres-17.pdf
  [Saved] reg-com_ahw_20260219_pres-18.pdf
  [Saved] reg-com_ahw_20260122_pres-14.pdf
  [Saved] reg-com_ahw_20260122_pres-15.pdf
  [Saved] reg-com_ahw_20260122_pres-16.pdf
  [Saved] reg-com_ahw_20260122_pres-17.pdf
  [Saved] reg-com_ahw_20260122_pres-18.pdf
  [

In [28]:
## =====================================================================
## PHASE 2: Configure Pipeline Processors
## =====================================================================
pipeline_options = PdfPipelineOptions()
pipeline_options.do_picture_description = True
pipeline_options.images_scale = 2.0
pipeline_options.generate_picture_images = True
pipeline_options.picture_description_options = PictureDescriptionVlmOptions(
    repo_id="HuggingFaceTB/SmolVLM-256M-Instruct",
    prompt=(
        "Analyze this image. If it is a plot, chart, or graph, transcribe its data points, "
        "axes, labels, and key trends into a clean Markdown table. "
        "If it is a regular picture, summarize it in one sentence."
    )
)

converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)}
)

SYSTEM_PROMPT = (
    "You are a precise data extraction agent. You will receive a document containing text "
    "interspersed with descriptions/tables of graphs that were embedded in the file. "
    "Your job is to parse this entire text and extract the core quantitative data into a "
    "single, structured JSON object. Respond ONLY with valid JSON."
)

## =====================================================================
## PHASE 3: Idempotent Processing (Globs directly from download folder)
## =====================================================================
# Scan the local download directory for any target PDFs
downloaded_files = [f for f in os.listdir(DOWNLOAD_DIR) if f.lower().endswith('.pdf')]

print(f"\n--- Processing Queue Ready: {len(downloaded_files)} target files detected locally ---")

for base_name in downloaded_files:
    pdf_path = os.path.join(DOWNLOAD_DIR, base_name)
    #FIXME currently the date downloaded and exact link is lost :/
    output_json_path = os.path.join(OUTPUT_DIR, base_name.replace(".pdf", "_extracted.txt"))

    # STATE CHECK 2: Skip AI pipeline execution if we already generated the JSON output before
    if os.path.exists(output_json_path):
        print(f"  [Complete] Extraction already exists for {base_name}. Skipping AI execution.")
        continue

    print(f"\nExecuting Pipeline for: {base_name}")
    try:
        # Run local VLM layout parsing
        result = converter.convert(pdf_path)

        # try 3 times to get ouptut
        unified_markdown = ""
        for i in range(3):
          unified_markdown = result.document.export_to_markdown()
          if unified_markdown.strip() != "":
            break

        text = unified_markdown
        # Extract structured data via local text LLM
        #response = ollama.chat(
        #    model="llama3",
        #    messages=[
        #        {"role": "system", "content": SYSTEM_PROMPT},
        #        {"role": "user", "content": f"Extract the data from this parsed text:\n\n{unified_markdown}"}
        #    ],
        #    options={"temperature": 0.0}
        #)
        with open(URLS_JSON, 'r') as f:
            data = {
                'extracted_md': text,
                'timestamp': time.time(),
                'url': json.loads(f.read())[base_name]

            }


        # Write out final structured results
        with open(output_json_path, "w", encoding="utf-8") as json_file:
            json_file.write(json.dumps(data))#response['message']['content'])

        print(f"  -> Successfully generated {os.path.basename(output_json_path)}")

    except Exception as e:
        print(f"  -> Pipeline failed on file {base_name}: {e}")

print("\n--- Pipeline Run Completed ---")


--- Processing Queue Ready: 31 target files detected locally ---

Executing Pipeline for: reg-com_ahw_20260219_pres-12.pdf


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[INFO] 2026-05-29 09:13:18,804 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-29 09:13:18,805 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-29 09:13:18,850 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-29 09:13:18,851 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-05-29 09:13:19,712 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-05-29 09:13:19,714 [RapidOCR] device_config.py:64: Using GPU device with ID: 0
[INFO] 2026-05-29 09:13:19,719 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-05-29 09:13:19,720 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

  -> Successfully generated reg-com_ahw_20260219_pres-12_extracted.txt

Executing Pipeline for: reg-com_ahw_20260324_pres-17.pdf
  -> Successfully generated reg-com_ahw_20260324_pres-17_extracted.txt

Executing Pipeline for: reg-com_ahw_20260122_pres-14.pdf
  -> Successfully generated reg-com_ahw_20260122_pres-14_extracted.txt

Executing Pipeline for: reg-com_ahw_20260122_pres-21.pdf


...

  -> Successfully generated reg-com_ahw_20260122_pres-19_extracted.txt

Executing Pipeline for: reg-com_ahw_20260219_pres-15.pdf


  -> Successfully generated reg-com_ahw_20260219_pres-15_extracted.txt

Executing Pipeline for: reg-com_ahw_20260324_pres-13.pdf


  -> Successfully generated reg-com_ahw_20260324_pres-13_extracted.txt

Executing Pipeline for: reg-com_ahw_20260219_pres-14.pdf
  -> Successfully generated reg-com_ahw_20260219_pres-14_extracted.txt

Executing Pipeline for: reg-com_ahw_20260324_pres-15.pdf
  -> Successfully generated reg-com_ahw_20260324_pres-15_extracted.txt

Executing Pipeline for: reg-com_ahw_20260122_pres-20.pdf


  -> Pipeline failed on file reg-com_ahw_20260122_pres-20.pdf: Pipeline StandardPdfPipeline failed

Executing Pipeline for: reg-com_ahw_20260122_pres-22.pdf


  -> Successfully generated reg-com_ahw_20260122_pres-22_extracted.txt

--- Pipeline Run Completed ---


In [32]:
!zip -r hpai.zip ./hpai_extractions/*

  adding: hpai_extractions/reg-com_ahw_20251215_pres-09_extracted.txt (deflated 73%)
  adding: hpai_extractions/reg-com_ahw_20251215_pres-10_extracted.txt (deflated 61%)
  adding: hpai_extractions/reg-com_ahw_20251215_pres-11_extracted.txt (deflated 63%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-14_extracted.txt (deflated 75%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-15_extracted.txt (deflated 68%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-16_extracted.txt (deflated 62%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-17_extracted.txt (deflated 62%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-18_extracted.txt (deflated 73%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-19_extracted.txt (deflated 65%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-21_extracted.txt (deflated 64%)
  adding: hpai_extractions/reg-com_ahw_20260122_pres-22_extracted.txt (deflated 66%)
  adding: hpai_extractions/reg-com_ahw_20260219_pres-09_extracted